In [3]:
import pandas as pd
import re
import pickle
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load dataset
df = pd.read_csv("train.csv", engine="python", on_bad_lines="skip")

# Clean text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text

df['comment_text'] = df['comment_text'].apply(clean_text)

X = df['comment_text']

# MULTI LABEL TARGET
y = df[['toxic','severe_toxic','obscene','threat','insult','identity_hate']]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Tokenization
MAX_WORDS = 20000
MAX_LEN = 150

tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN)
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN)

# Model
model = Sequential()

model.add(Embedding(MAX_WORDS,128,input_length=MAX_LEN))

model.add(Bidirectional(LSTM(64)))

model.add(Dropout(0.5))

# 6 OUTPUTS
model.add(Dense(6,activation="sigmoid"))

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

model.fit(
    X_train_pad,
    y_train,
    epochs=3,
    batch_size=64,
    validation_data=(X_test_pad,y_test)
)

# Save model
model.save("Toxicity_model.h5")

# Save tokenizer
pickle.dump(tokenizer,open("Tokenizer.pkl","wb"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 431s 380ms/step - accuracy: 0.7170 - loss: 0.1338 - val_accuracy: 0.9933 - val_loss: 0.0559
Epoch 2/3
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 428s 381ms/step - accuracy: 0.9706 - loss: 0.0546 - val_accuracy: 0.9933 - val_loss: 0.0559
Epoch 3/3
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 439s 391ms/step - accuracy: 0.9791 - loss: 0.0465 - val_accuracy: 0.9933 - val_loss: 0.0525
